<a href="https://colab.research.google.com/github/scardenol/Stochastic_Optimization_2026/blob/main/Algoritmos/clase4_Integer_LShaped_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integer L-Shaped: extensión entera del método L-Shaped

Este notebook **extiende** la implementación de `clase3_LShaped_algorithm.ipynb` (método
L-Shaped para primera etapa **continua**) al caso en que las variables de primera etapa
$x$ son **enteras/binarias**, siguiendo las notas de *Optimización Estocástica — Clase 4*.

**Idea central (Integer L-Shaped, Laporte & Louveaux):** se combina

1. un esquema de **Branch-and-Bound** sobre $x\in X\subseteq\mathbb{Z}^{n_1}$ (relajación continua $0\le \bar X$, ramificación piso/techo), con
2. la **descomposición L-shaped** dentro de cada nodo para aproximar $\mathcal{Q}(x)=\mathbb{E}_\xi[Q(x,\xi)]$ mediante $\theta$,

y cuando la solución de un nodo resulta entera:

- si el recurso es **continuo**, se genera el **corte de Benders estándar** (igual que en Clase 3) a partir de los duales;
- si el recurso es **entero** (no hay duales disponibles), se genera un **corte de integralidad** (Laporte & Louveaux) usando el valor exacto $Q(x^\nu)$ y una cota inferior global $L\le Q(x)\ \forall x$:

$$\theta \ge Q(x^\nu) - \big(Q(x^\nu)-L\big)\Big(\sum_{i\in S}(1-x_i)+\sum_{i\notin S}x_i\Big), \qquad S=\{i: x^\nu_i=1\}.$$

Ambos cortes son **globales**: se agregan al pool compartido por todos los nodos del árbol.

Se valida la implementación con:
1. El **Ejemplo 1** de las notas de Clase 4 (SIP binario con recurso entero) — reproduciendo a mano el corte de integralidad.
2. La **Actividad 2** de las notas (red de carga EV, binario con recurso continuo) — una iteración manual desde $x^\nu=(1,0)$ y luego el método completo.
3. La **Actividad 3** (cuadrillas contra baches en Chicago) usando la plantilla del profesor, con $x_j\in\{0,\dots,3\}$ (entero general, no binario).

## Cargar paquetes

In [1]:
!pip install gurobipy -q  # instalar gurobipy, si no está instalado
import numpy as np
import heapq
import gurobipy as gp
from gurobipy import GRB

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 53.4 MB/s eta 0:00:00


## 1. Estructura de datos del problema (extendida a primera etapa entera)

Se extiende `TwoStageProblem` de la Clase 3 con:

- `x_lb`, `x_ub`: cotas de las variables de primera etapa (permiten dominios enteros generales, no solo binarios).
- `x_int`: si `True`, $x$ es entera (activa el Branch-and-Bound y, si además `x_ub<=1`, el corte de integralidad binario).
- `recourse_int` (y `y_int` por escenario): si el recurso $y_k$ es entero. En tal caso el subproblema se resuelve como MIP y **no** se obtienen duales, por lo que solo se puede usar el corte de integralidad (no el de Benders).

In [2]:
class TwoStageProblem:
    """Contenedor de datos del problema de dos etapas. Extiende la clase de la
    Clase 3 para soportar variables de primera etapa ENTERAS y recurso entero."""

    def __init__(self, c, A, b, x_lb=None, x_ub=None, x_int=True, recourse_int=False):
        self.c = np.atleast_1d(np.asarray(c, dtype=float))
        self.A = np.atleast_2d(np.asarray(A, dtype=float))
        self.b = np.atleast_1d(np.asarray(b, dtype=float))
        self.n1 = self.c.shape[0]
        self.x_lb = np.zeros(self.n1) if x_lb is None else np.atleast_1d(np.asarray(x_lb, dtype=float))
        self.x_ub = (np.full(self.n1, GRB.INFINITY) if x_ub is None
                     else np.atleast_1d(np.asarray(x_ub, dtype=float)))
        self.x_int = x_int                # x_i enteras (dominio de 1ra etapa)
        self.recourse_int = recourse_int  # y_k enteras por defecto (recurso entero)
        self.scenarios = []               # lista de dicts: p, q, W, T, h, y_int

    def add_scenario(self, p, q, W, T, h, y_int=None):
        """Agrega un escenario k con su probabilidad p y datos (q, W, T, h)."""
        self.scenarios.append({
            "p": p,
            "q": np.atleast_1d(np.asarray(q, dtype=float)),
            "W": np.atleast_2d(np.asarray(W, dtype=float)),
            "T": np.atleast_2d(np.asarray(T, dtype=float)),
            "h": np.atleast_1d(np.asarray(h, dtype=float)),
            "y_int": self.recourse_int if y_int is None else y_int,
        })

## 2. Algoritmo Integer L-Shaped

Siguiendo el flujo general de las notas (*Integer L-Shaped: Flujo del algoritmo*):

- **Paso 0:** lista de nodos activos = {nodo raíz = $(x_{lb}, x_{ub})$ original}, incumbente $\bar z=\infty$. Se calcula una cota inferior global $L=\min_{x\in[x_{lb},x_{ub}]}\mathbb{E}_\xi[C(x,\xi)]$ resolviendo **un único LP conjunto** (relajación continua de $x$ y de $y$), tal como lo sugieren las notas.
- **Mientras** la lista no esté vacía:
  1. **Seleccionar nodo** (estrategia *mejor cota*: se explora primero el nodo con menor cota inferior).
  2. **Resolver el maestro** $\min\{c^Tx+\theta \mid Ax=b,\ x_{lb}^v\le x\le x_{ub}^v,\ \text{cortes globales}\}$ → $(x^\nu,\theta^\nu)$.
  3. **Podar por cota:** si el valor del nodo $\ge \bar z$, se descarta.
  4. **Si $x^\nu$ es fraccional:** se ramifica sobre la variable más fraccional $x_j$, creando los hijos $x_j\le\lfloor x_j^\nu\rfloor$ y $x_j\ge\lceil x_j^\nu\rceil$.
  5. **Si $x^\nu$ es entera:** se evalúa el recurso real $Q(x^\nu)=\sum_k p_k Q(x^\nu,\xi_k)$ (resolviendo cada subproblema como LP o MIP según `y_int`) y $z_{real}=c^Tx^\nu+Q(x^\nu)$; se actualiza el incumbente si mejora.
     - Si $\theta^\nu \ge Q(x^\nu)$: el nodo queda cerrado (la aproximación ya es exacta ahí).
     - Si no: se agrega **globalmente** el corte de Benders (si hay duales) y/o el corte de integralidad (si $x$ es binaria), y el nodo se reoptimiza (no se poda automáticamente).
- **Fin:** cuando la lista de nodos está vacía, el incumbente es la solución óptima.

In [3]:
class IntegerLShapedSolver:
    """Integer L-Shaped: Branch-and-Bound sobre x (entero) + descomposición L-shaped
    (cortes de Benders) en cada nodo, con cortes de integralidad (Laporte & Louveaux)
    cuando el recurso es entero y por tanto no hay duales disponibles."""

    def __init__(self, problem, theta_lb=-1e7, tol=1e-6, max_nodes=2000, verbose=True):
        self.problem = problem
        self.theta_lb = theta_lb
        self.tol = tol
        self.max_nodes = max_nodes
        self.verbose = verbose

        self.optimality_cuts = []   # cortes de Benders (E, e): theta >= e - E@x      [globales]
        self.integrality_cuts = []  # cortes de integralidad (S, qS, L)               [globales]
        self.L_global = None        # cota inferior global de Q(x)

    # ---------------- Cota inferior global L ----------------
    def compute_L(self):
        """L = min_x E_xi[C(x,xi)] sobre la relajación continua conjunta de (x,y).
        Como C(x) es convexa y se busca su mínimo sobre una caja, basta un único LP
        que optimice x y todos los y_k simultáneamente (no requiere descomposición)."""
        p = self.problem
        m = gp.Model("cota_L")
        m.Params.OutputFlag = 0
        x = m.addMVar(p.n1, lb=p.x_lb, ub=p.x_ub, name="x")
        obj = 0
        for k, s in enumerate(p.scenarios):
            n2 = s["W"].shape[1]
            y = m.addMVar(n2, lb=0, name=f"y_{k}")
            m.addConstr(s["W"] @ y == s["h"] - s["T"] @ x)
            obj += s["p"] * (s["q"] @ y)
        m.setObjective(obj, GRB.MINIMIZE)
        m.optimize()
        if m.Status != GRB.OPTIMAL:
            raise RuntimeError("No fue posible calcular la cota L (LP infactible/no acotado).")
        return m.ObjVal

    # ---------------- Maestro (LP relajado del nodo) ----------------
    def _solve_master(self, lb_v, ub_v):
        p = self.problem
        m = gp.Model("maestro")
        m.Params.OutputFlag = 0
        x = m.addMVar(p.n1, lb=lb_v, ub=ub_v, name="x")
        theta = m.addVar(lb=self.theta_lb, name="theta")
        m.addConstr(p.A @ x == p.b)

        for E, e in self.optimality_cuts:
            m.addConstr(theta >= e - E @ x)

        for S, qS, L in self.integrality_cuts:
            n = p.n1
            not_S = [i for i in range(n) if i not in S]
            delta = gp.LinExpr()
            for i in S:
                delta += (1 - x[i])
            for i in not_S:
                delta += x[i]
            m.addConstr(theta >= qS - (qS - L) * delta)

        m.setObjective(p.c @ x + theta, GRB.MINIMIZE)
        m.optimize()
        if m.Status != GRB.OPTIMAL:
            return None, None, None
        return np.array(x.X), theta.X, m.ObjVal

    # ---------------- Subproblema de recurso ----------------
    def _solve_subproblem(self, k, x_val):
        s = self.problem.scenarios[k]
        rhs = s["h"] - s["T"] @ x_val
        m = gp.Model(f"sub_{k}")
        m.Params.OutputFlag = 0
        n2 = s["W"].shape[1]
        if s["y_int"]:
            y = m.addMVar(n2, lb=0, vtype=GRB.INTEGER, name="y")
        else:
            y = m.addMVar(n2, lb=0, name="y")
        con = m.addConstr(s["W"] @ y == rhs)
        m.setObjective(s["q"] @ y, GRB.MINIMIZE)
        m.optimize()
        if m.Status != GRB.OPTIMAL:
            return False, None, None
        pi = None if s["y_int"] else np.array(con.Pi)   # duales solo si el recurso es continuo
        return True, m.ObjVal, pi

    def _evaluate_Qx(self, x_val):
        """Evalúa Q(x)=sum_k p_k Q(x,xi_k) exactamente y arma el corte de Benders
        estándar si TODOS los subproblemas entregaron duales (recurso continuo)."""
        p = self.problem
        Qx = 0.0
        duals_ok = True
        duals, hs, Ts, ps = [], [], [], []
        for k, s in enumerate(p.scenarios):
            feas, val, pi = self._solve_subproblem(k, x_val)
            if not feas:
                raise RuntimeError("Subproblema infactible: este notebook no implementa "
                                    "cortes de factibilidad (ver Clase 3 para esa extensión).")
            Qx += s["p"] * val
            if pi is None:
                duals_ok = False
            else:
                duals.append(pi); hs.append(s["h"]); Ts.append(s["T"]); ps.append(s["p"])
        benders_cut = None
        if duals_ok:
            e = sum(pk * (pik @ hk) for pk, pik, hk in zip(ps, duals, hs))
            E = sum(pk * (pik @ Tk) for pk, pik, Tk in zip(ps, duals, Ts))
            benders_cut = (E, e)
        return Qx, benders_cut

    def _is_integer(self, x_val):
        return np.all(np.abs(x_val - np.round(x_val)) <= 1e-6)

    def solve(self):
        p = self.problem
        if self.L_global is None:
            self.L_global = self.compute_L()
            if self.verbose:
                print(f"Cota inferior global L = {self.L_global:.6f}")

        root = (p.x_lb.copy(), p.x_ub.copy())
        active = [(0.0, 0, root)]   # heap: (cota LP del nodo, contador, (lb_v, ub_v))
        counter = 1
        incumbent = {"obj": np.inf, "x": None, "theta": None}
        node_log = []
        nodes_processed = 0

        while active and nodes_processed < self.max_nodes:
            _, _, (lb_v, ub_v) = heapq.heappop(active)
            nodes_processed += 1

            x_val, theta_val, node_obj = self._solve_master(lb_v, ub_v)
            if x_val is None:
                continue  # nodo infactible

            if node_obj >= incumbent["obj"] - self.tol:
                if self.verbose:
                    print(f"Nodo {nodes_processed}: podado por cota "
                          f"(LB nodo={node_obj:.4f} >= UB={incumbent['obj']:.4f})")
                continue

            if not self._is_integer(x_val):
                frac = np.abs(x_val - np.round(x_val))
                j = int(np.argmax(frac))
                xj = x_val[j]
                lb1, ub1 = lb_v.copy(), ub_v.copy(); ub1[j] = np.floor(xj)
                lb2, ub2 = lb_v.copy(), ub_v.copy(); lb2[j] = np.ceil(xj)
                if self.verbose:
                    print(f"Nodo {nodes_processed}: x={np.round(x_val,4)} fraccional "
                          f"(x_{j}={xj:.4f}), LB={node_obj:.4f} -> ramifica")
                counter += 1; heapq.heappush(active, (node_obj, counter, (lb1, ub1)))
                counter += 1; heapq.heappush(active, (node_obj, counter, (lb2, ub2)))
                continue

            # x^v es entera: evaluar el recurso real
            x_int_val = np.round(x_val)
            Qx, benders_cut = self._evaluate_Qx(x_int_val)
            z_real = float(p.c @ x_int_val + Qx)
            node_log.append({"node": nodes_processed, "x": x_int_val.copy(),
                              "theta": theta_val, "Qx": Qx, "z_real": z_real})

            if self.verbose:
                print(f"Nodo {nodes_processed}: x={x_int_val}, theta={theta_val:.4f}, "
                      f"Q(x)={Qx:.4f}, z_real={z_real:.4f}")

            if z_real < incumbent["obj"] - self.tol:
                incumbent = {"obj": z_real, "x": x_int_val.copy(), "theta": theta_val}
                if self.verbose:
                    print(f"   -> nuevo incumbente: x={x_int_val}, obj={z_real:.4f}")

            if theta_val >= Qx - self.tol:
                continue  # el nodo se cierra por integralidad (aproximación exacta aquí)

            added_any = False
            if benders_cut is not None:
                self.optimality_cuts.append(benders_cut)
                added_any = True
            if p.x_int and np.all(p.x_ub <= 1 + 1e-9):
                S = set(int(i) for i in np.where(x_int_val > 0.5)[0])
                self.integrality_cuts.append((S, Qx, self.L_global))
                added_any = True
            if not added_any:
                raise RuntimeError("No se pudo fortalecer el nodo: se necesita recurso "
                                    "continuo (duales) o dominio binario para x.")

            counter += 1
            heapq.heappush(active, (node_obj, counter, (lb_v, ub_v)))  # reoptimizar el mismo nodo

        return {"x": incumbent["x"], "obj": incumbent["obj"], "nodes": nodes_processed,
                "log": node_log, "L": self.L_global}

## 3. Validación 1: Ejemplo 1 de las notas (SIP binario, recurso entero)

Del PDF de Clase 4 (*Corte de Integralidad: Ejemplo 1*):

$$\min_{x\in\{0,1\}^2} \mathcal{Q}(x), \quad
Q(x,\xi)=\min\{-2y_1-3y_2 \mid y_1+2y_2\le \xi_1-x_1,\ y_1\le \xi_2-x_2,\ y_1,y_2\ge0 \text{ enteros}\}$$

con $\xi^1=(2,2)^T$, $\xi^2=(4,3)^T$, $p_k=0.5$. Se convierten las desigualdades a igualdad con holguras
enteras $s_1,s_2\ge0$: $y=[y_1,y_2,s_1,s_2]$.

In [4]:
problem_e1 = TwoStageProblem(c=[0.0, 0.0], A=[[0.0, 0.0]], b=[0.0],
                              x_lb=[0, 0], x_ub=[1, 1], x_int=True, recourse_int=True)

W_e1 = [[1.0, 2.0, 1.0, 0.0],
        [1.0, 0.0, 0.0, 1.0]]
T_e1 = [[1.0, 0.0],
        [0.0, 1.0]]
for (xi1, xi2), p in [((2.0, 2.0), 0.5), ((4.0, 3.0), 0.5)]:
    problem_e1.add_scenario(p=p, q=[-2.0, -3.0, 0.0, 0.0], W=W_e1, T=T_e1, h=[xi1, xi2])

solver_e1 = IntegerLShapedSolver(problem_e1, theta_lb=-100, verbose=False)

### 3.1 Reproducir a mano la iteración del profesor en $x^\nu=(0,1)^T$

Las notas obtienen $L=-5.75$ (cota inferior global), $q_S=\mathcal{Q}(0,1)=-5$ y el corte

$$\theta \ge 0.75\,(x_2-x_1) - 5.75.$$

Verificamos que nuestra implementación reproduce exactamente estos tres números.

In [5]:
L_e1 = solver_e1.compute_L()
xv = np.array([0.0, 1.0])
Qx_e1, _ = solver_e1._evaluate_Qx(xv)   # recurso entero -> sin duales, solo corte de integralidad
qS, S = Qx_e1, {1}   # S = {i : x_i^v = 1} = {1} (x2)

E_coef = np.array([1.0 if i not in S else -1.0 for i in range(2)]) * (qS - L_e1)
# theta >= qS - (qS-L)*[(1-x2)+x1] = qS-(qS-L) - (qS-L)x1 + (qS-L)x2
const = qS - (qS - L_e1)

print(f"L                = {L_e1:.4f}   (notas: -5.75)")
print(f"q_S = Q(x^v)     = {qS:.4f}   (notas: -5)")
print(f"Corte:  theta >= {(qS-L_e1):.4f}*(x2 - x1) + {const:.4f}   (notas: 0.75*(x2-x1) - 5.75)")

Restricted license - for non-production use only - expires 2027-11-29
L                = -5.7500   (notas: -5.75)
q_S = Q(x^v)     = -5.0000   (notas: -5)
Corte:  theta >= 0.7500*(x2 - x1) + -5.7500   (notas: 0.75*(x2-x1) - 5.75)


### 3.2 Resolver el problema completo y validar contra enumeración exhaustiva

In [6]:
res_e1 = IntegerLShapedSolver(problem_e1, theta_lb=-100, verbose=True).solve()
print("\n=== Integer L-Shaped ===")
print(f"x* = {res_e1['x']}   obj* = {res_e1['obj']}   nodos explorados = {res_e1['nodes']}")

print("\n=== Enumeración exhaustiva (solo 4 puntos binarios) ===")
best = None
for x1 in [0, 1]:
    for x2 in [0, 1]:
        Qx, _ = solver_e1._evaluate_Qx(np.array([x1, x2], dtype=float))
        print(f"x=({x1},{x2}) -> Q(x) = {Qx}")
        if best is None or Qx < best[1]:
            best = ((x1, x2), Qx)
print(f"\nÓptimo por enumeración: x*={best[0]}, obj*={best[1]}")
assert np.allclose(res_e1["x"], best[0]) and abs(res_e1["obj"] - best[1]) < 1e-6, "¡No coincide!"
print("\n✅ Coincide exactamente con la enumeración exhaustiva.")

Cota inferior global L = -5.750000
Nodo 1: x=[0. 0.], theta=-100.0000, Q(x)=-5.5000, z_real=-5.5000
   -> nuevo incumbente: x=[0. 0.], obj=-5.5000
Nodo 2: x=[1. 1.], theta=-6.0000, Q(x)=-3.5000, z_real=-3.5000
Nodo 3: x=[1. 0.], theta=-5.7500, Q(x)=-4.0000, z_real=-4.0000
Nodo 4: x=[0.5 0.5] fraccional (x_0=0.5000), LB=-5.7500 -> ramifica
Nodo 5: x=[0. 1.], theta=-5.7500, Q(x)=-5.0000, z_real=-5.0000
Nodo 6: podado por cota (LB nodo=-4.7656 >= UB=-5.5000)
Nodo 7: x=[0.   0.25] fraccional (x_1=0.2500), LB=-5.5625 -> ramifica
Nodo 8: podado por cota (LB nodo=-5.5000 >= UB=-5.5000)
Nodo 9: podado por cota (LB nodo=-5.0000 >= UB=-5.5000)

=== Integer L-Shaped ===
x* = [0. 0.]   obj* = -5.5   nodos explorados = 9

=== Enumeración exhaustiva (solo 4 puntos binarios) ===
x=(0,0) -> Q(x) = -5.5
x=(0,1) -> Q(x) = -5.0
x=(1,0) -> Q(x) = -4.0
x=(1,1) -> Q(x) = -3.5

Óptimo por enumeración: x*=(0, 0), obj*=-5.5

✅ Coincide exactamente con la enumeración exhaustiva.


## 4. Validación 2: Actividad 2 de las notas (red de carga para vehículos eléctricos)

Recordando el enunciado: construir en $A$ cuesta \$20k (capacidad 40 MWh a \$1k/MWh) y en $B$ cuesta
\$16k (capacidad 25 MWh a \$1.2k/MWh); la demanda faltante se compra a la red a \$3k/MWh. Demanda
$\xi\in\{30,50\}$ MWh con $p_k=0.5$. Aquí $x=(x_A,x_B)$ es binaria pero el **recurso es continuo**, por lo
que se usará el **corte de Benders estándar** (no el de integralidad).

In [7]:
c_ev = [20.0, 16.0]
problem_ev = TwoStageProblem(c=c_ev, A=[[0.0, 0.0]], b=[0.0],
                              x_lb=[0, 0], x_ub=[1, 1], x_int=True, recourse_int=False)

# y = [yA, yB, yP, capslackA, capslackB]
W_ev = [[1.0, 1.0, 1.0, 0.0, 0.0],    # balance de demanda: yA+yB+yP = xi
        [1.0, 0.0, 0.0, 1.0, 0.0],    # capacidad A: yA + capslackA = 40 xA
        [0.0, 1.0, 0.0, 0.0, 1.0]]    # capacidad B: yB + capslackB = 25 xB
T_ev = [[0.0, 0.0],
        [-40.0, 0.0],
        [0.0, -25.0]]
q_ev = [1.0, 1.2, 3.0, 0.0, 0.0]
for xi, p in [(30.0, 0.5), (50.0, 0.5)]:
    problem_ev.add_scenario(p=p, q=q_ev, W=W_ev, T=T_ev, h=[xi, 0.0, 0.0])

solver_ev = IntegerLShapedSolver(problem_ev, theta_lb=-1e5, verbose=True)

### 4.1 Una iteración manual partiendo de $x^\nu=(1,0)^T$ (construir solo en A)

Con demanda baja (30) A cubre todo: $20+30=50$. Con demanda alta (50) A cubre 40 y compra 10 a la red:
$20+40+30=90$. Costo esperado $=0.5(50)+0.5(90)=70$.

In [8]:
xv_ev = np.array([1.0, 0.0])
L_ev = solver_ev.compute_L()
Qx_ev, benders_cut_ev = solver_ev._evaluate_Qx(xv_ev)
E_ev, e_ev = benders_cut_ev

print(f"Cota global L        = {L_ev:.4f}")
print(f"Q(x^v)               = {Qx_ev:.4f}")
print(f"Corte de Benders:  theta >= {e_ev:.4f} - ({E_ev[0]:.4f})xA - ({E_ev[1]:.4f})xB")
print(f"z_real(x^v=(1,0))    = {c_ev[0]*xv_ev[0] + c_ev[1]*xv_ev[1] + Qx_ev:.4f}  (esperado: 70)")

Cota global L        = 41.0000
Q(x^v)               = 50.0000
Corte de Benders:  theta >= 90.0000 - (40.0000)xA - (22.5000)xB
z_real(x^v=(1,0))    = 70.0000  (esperado: 70)


### 4.2 Resolver el método completo y validar contra enumeración exhaustiva

In [9]:
res_ev = IntegerLShapedSolver(problem_ev, theta_lb=-1e5, verbose=True).solve()
print("\n=== Integer L-Shaped ===")
print(f"x* = {res_ev['x']}   obj* = {res_ev['obj']}   nodos explorados = {res_ev['nodes']}")

print("\n=== Enumeración exhaustiva ===")
best_ev = None
for xA in [0, 1]:
    for xB in [0, 1]:
        Qx, _ = solver_ev._evaluate_Qx(np.array([xA, xB], dtype=float))
        z = c_ev[0]*xA + c_ev[1]*xB + Qx
        print(f"x=({xA},{xB}) -> costo total esperado = {z}")
        if best_ev is None or z < best_ev[1]:
            best_ev = ((xA, xB), z)
print(f"\nÓptimo por enumeración: x*={best_ev[0]}, obj*={best_ev[1]}")
assert np.allclose(res_ev["x"], best_ev[0]) and abs(res_ev["obj"] - best_ev[1]) < 1e-6, "¡No coincide!"
print("\n✅ Coincide exactamente con la enumeración exhaustiva.")

Cota inferior global L = 41.000000
Nodo 1: x=[0. 0.], theta=-100000.0000, Q(x)=120.0000, z_real=120.0000
   -> nuevo incumbente: x=[0. 0.], obj=120.0000
Nodo 2: x=[1. 1.], theta=-5.0000, Q(x)=41.0000, z_real=77.0000
   -> nuevo incumbente: x=[1. 1.], obj=77.0000
Nodo 3: x=[0.97   0.0285] fraccional (x_0=0.9700), LB=60.9757 -> ramifica
Nodo 4: podado por cota (LB nodo=91.0000 >= UB=77.0000)
Nodo 5: x=[1. 0.], theta=41.0000, Q(x)=50.0000, z_real=70.0000
   -> nuevo incumbente: x=[1. 0.], obj=70.0000
Nodo 6: podado por cota (LB nodo=70.0000 >= UB=70.0000)

=== Integer L-Shaped ===
x* = [1. 0.]   obj* = 70.0   nodos explorados = 6

=== Enumeración exhaustiva ===
x=(0,0) -> costo total esperado = 120.0
x=(0,1) -> costo total esperado = 91.0
x=(1,0) -> costo total esperado = 70.0
x=(1,1) -> costo total esperado = 77.0

Óptimo por enumeración: x*=(1, 0), obj*=70.0

✅ Coincide exactamente con la enumeración exhaustiva.


## 5. Aplicación: Actividad 3 — cuadrillas para baches en Chicago

Aquí $x_j\in\{0,1,2,3\}$ (cuadrillas por base) ya **no es binaria**, así que el corte de integralidad de
Laporte & Louveaux (que depende de $S=\{i:x_i=1\}$) no aplica. El código lo detecta automáticamente:
como `x_ub` no es $\le 1$, `IntegerLShapedSolver` **no** genera cortes de integralidad y se apoya
únicamente en el **corte de Benders + ramificación piso/techo** (recurso continuo → siempre hay duales).
Esto sigue siendo *Integer L-Shaped* en el sentido general descrito en las notas (Branch-and-Bound +
descomposición L-shaped en cada nodo), solo que sin el corte adicional específico para dominio binario.

Se reutiliza la función `preparar_datos()` de la plantilla del profesor y se completa `resolver_modelo()`
usando `IntegerLShapedSolver`.

In [10]:
from urllib.parse import urlencode
import pandas as pd
from sklearn.cluster import KMeans

ANIO = 2024
N_ZONAS = 6
CAPACIDAD = 600.0
COSTO_CUADRILLA = 80_000.0
COSTO_KM_SOLICITUD = 8.0
COSTO_EMERGENCIA = 450.0
MAX_CUADRILLAS_ZONA = 3


def preparar_datos():
    base = "https://data.cityofchicago.org/resource/v6vf-nfxy.csv"
    filtro = (
        "sr_short_code='PHF' "
        "AND created_date >= '2024-01-01T00:00:00' "
        "AND created_date < '2025-01-01T00:00:00' "
        "AND community_area IS NOT NULL "
        "AND latitude IS NOT NULL AND longitude IS NOT NULL "
        "AND duplicate=false"
    )
    url = base + "?" + urlencode(
        {
            "$select": "community_area,created_date,created_month,latitude,longitude",
            "$where": filtro,
            "$limit": 50000,
        }
    )
    datos = pd.read_csv(url, parse_dates=["created_date"])

    areas = (
        datos.groupby("community_area")
        .agg(latitude=("latitude", "mean"), longitude=("longitude", "mean"),
             solicitudes_anuales=("created_date", "size"))
        .reset_index()
    )
    km = KMeans(n_clusters=N_ZONAS, random_state=42, n_init=20)
    areas["cluster"] = km.fit_predict(areas[["latitude", "longitude"]],
                                       sample_weight=areas["solicitudes_anuales"])

    centros_crudos = (
        areas.groupby("cluster")
        .apply(lambda g: pd.Series({
            "latitude": np.average(g.latitude, weights=g.solicitudes_anuales),
            "longitude": np.average(g.longitude, weights=g.solicitudes_anuales),
        }), include_groups=False)
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )
    renombrar = {cluster: z for z, cluster in enumerate(centros_crudos.index)}
    areas["zona"] = areas["cluster"].map(renombrar).astype(int)
    datos = datos.merge(areas[["community_area", "zona"]], on="community_area")

    demanda = (
        datos.groupby(["created_month", "zona"]).size().unstack(fill_value=0)
        .reindex(index=range(1, 13), columns=range(N_ZONAS), fill_value=0).astype(float)
    )
    demanda.index = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                      "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
    demanda.columns = [f"Z{j + 1}" for j in range(N_ZONAS)]

    centros = (datos.groupby("zona").agg(latitude=("latitude", "mean"), longitude=("longitude", "mean"))
               .reindex(range(N_ZONAS)))
    lat = np.radians(centros.latitude.to_numpy())
    lon = np.radians(centros.longitude.to_numpy())
    a = (np.sin((lat[:, None] - lat[None, :]) / 2) ** 2
         + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin((lon[:, None] - lon[None, :]) / 2) ** 2)
    distancias = 2 * 6371.0 * np.arcsin(np.sqrt(a))
    return datos, areas, demanda, centros, distancias, url


datos, areas, demanda, centros, distancias, url = preparar_datos()
print(f"Filas descargadas: {len(datos):,}")
print(f"Áreas comunitarias: {datos.community_area.nunique()}")
print("\nEscenarios de demanda (solicitudes por mes y zona):")
print(demanda.astype(int))

Filas descargadas: 36,486
Áreas comunitarias: 77

Escenarios de demanda (solicitudes por mes y zona):
       Z1    Z2    Z3    Z4   Z5   Z6
Ene  1100   749  1174  1032  714  674
Feb  1578  1123  1460  1361  857  623
Mar  1136   815  1176   779  632  641
Abr   943   683  1129   618  579  565
May   572   532   706   448  428  406
Jun   440   402   562   498  303  328
Jul   472   334   518   372  218  246
Ago   311   229   482   304  166  162
Sep   313   248   410   215  151  158
Oct   270   172   355   209  162  122
Nov   241   147   333   168  123  114
Dic   197   133   223   140  150  152


### 5.1 Formulación como `TwoStageProblem` y `resolver_modelo()`

Variables de recurso por escenario $s$: $y_{ij}\ge0$ (solicitudes de $i$ atendidas desde $j$),
$u_i\ge0$ (atendidas por emergencia) y una holgura de capacidad $\text{capslack}_j\ge0$:

- Capacidad: $\sum_i y_{s,i,j} + \text{capslack}_{s,j} = 600\,x_j\quad\forall j$
- Demanda:   $\sum_j y_{s,i,j} + u_{s,i} = d_{s,i}\quad\forall i$

$$\text{Objetivo:}\quad 80000\sum_j x_j + \sum_s p_s\Big(8\sum_{i,j} r_{ij}\,y_{s,i,j} + 450\sum_i u_{s,i}\Big)$$

In [11]:
def resolver_modelo(demanda, distancias, probabilidades=None, x_fijo=None, verbose=True):
    """Resuelve el equivalente estocástico de dos etapas con Integer L-Shaped.

    Retorna: costo óptimo, x* (cuadrillas por base), y matriz u* (solicitudes de
    emergencia) apiladas por escenario (S x N_ZONAS).
    """
    S = demanda.shape[0]
    n_zonas = demanda.shape[1]
    if probabilidades is None:
        probabilidades = np.full(S, 1.0 / S)

    n2 = n_zonas * n_zonas + n_zonas + n_zonas   # y_ij + u_i + capslack_j

    problem = TwoStageProblem(
        c=[COSTO_CUADRILLA] * n_zonas, A=[[0.0] * n_zonas], b=[0.0],
        x_lb=[0] * n_zonas, x_ub=[MAX_CUADRILLAS_ZONA] * n_zonas,
        x_int=True, recourse_int=False,
    )

    dist = distancias.to_numpy() if hasattr(distancias, "to_numpy") else np.asarray(distancias)
    dem = demanda.to_numpy() if hasattr(demanda, "to_numpy") else np.asarray(demanda)

    for s in range(S):
        W = np.zeros((2 * n_zonas, n2))
        T = np.zeros((2 * n_zonas, n_zonas))
        q = np.zeros(n2)
        for j in range(n_zonas):
            for i in range(n_zonas):
                W[j, i * n_zonas + j] = 1.0
                q[i * n_zonas + j] = COSTO_KM_SOLICITUD * dist[i, j]
            W[j, n_zonas * n_zonas + n_zonas + j] = 1.0   # capslack_j
            T[j, j] = -CAPACIDAD
        for i in range(n_zonas):
            row = n_zonas + i
            for j in range(n_zonas):
                W[row, i * n_zonas + j] = 1.0
            W[row, n_zonas * n_zonas + i] = 1.0            # u_i
            q[n_zonas * n_zonas + i] = COSTO_EMERGENCIA
        h = np.concatenate([np.zeros(n_zonas), dem[s]])
        problem.add_scenario(p=probabilidades[s], q=q, W=W, T=T, h=h)

    if x_fijo is not None:
        # Solo evaluar el costo de una decision x fija (para comparar con Tarea 6)
        solver = IntegerLShapedSolver(problem, theta_lb=-1e9, verbose=False)
        Qx, _ = solver._evaluate_Qx(np.asarray(x_fijo, dtype=float))
        costo = float(np.dot(problem.c, x_fijo)) + Qx
        return costo, np.asarray(x_fijo, dtype=float), None

    solver = IntegerLShapedSolver(problem, theta_lb=-1e9, verbose=verbose)
    res = solver.solve()

    # reconstruir u* resolviendo cada subproblema en el x optimo
    u_matrix = np.zeros((S, n_zonas))
    for s in range(S):
        sdata = problem.scenarios[s]
        rhs = sdata["h"] - sdata["T"] @ res["x"]
        m = gp.Model(f"recon_{s}"); m.Params.OutputFlag = 0
        y = m.addMVar(n2, lb=0, name="y")
        m.addConstr(sdata["W"] @ y == rhs)
        m.setObjective(sdata["q"] @ y, GRB.MINIMIZE)
        m.optimize()
        u_matrix[s, :] = y.X[n_zonas * n_zonas: n_zonas * n_zonas + n_zonas]

    return res["obj"], res["x"], u_matrix

### 5.2 Resolver con los 12 escenarios (meses) reales de 2024

In [12]:
costo_opt, x_opt, u_opt = resolver_modelo(demanda, distancias, verbose=True)

print("\n=== Resultado ===")
print("Cuadrillas por zona x* =", x_opt.astype(int))
print("Total de cuadrillas    =", int(x_opt.sum()))
print(f"Costo esperado óptimo  = ${costo_opt:,.2f}")

Cota inferior global L = 0.000000
Nodo 1: x=[0. 0. 0. 0. 0. 0.], theta=-1000000000.0000, Q(x)=1368225.0000, z_real=1368225.0000
   -> nuevo incumbente: x=[0. 0. 0. 0. 0. 0.], obj=1368225.0000
Nodo 2: x=[3. 3. 3. 3. 3. 3.], theta=-3491775.0000, Q(x)=0.0000, z_real=1440000.0000
Nodo 3: x=[0.     2.0675 3.     0.     0.     0.    ] fraccional (x_1=2.0675), LB=405400.0000 -> ramifica
Nodo 4: x=[3.     0.     2.0675 0.     0.     0.    ] fraccional (x_2=2.0675), LB=405400.0000 -> ramifica
Nodo 5: x=[2.0675 3.     0.     0.     0.     0.    ] fraccional (x_0=2.0675), LB=405400.0000 -> ramifica
Nodo 6: x=[3.     0.0675 2.     0.     0.     0.    ] fraccional (x_1=0.0675), LB=405400.0000 -> ramifica
Nodo 7: x=[2.0675 0.     3.     0.     0.     0.    ] fraccional (x_0=2.0675), LB=405400.0000 -> ramifica
Nodo 8: x=[0.     3.     2.0675 0.     0.     0.    ] fraccional (x_2=2.0675), LB=405400.0000 -> ramifica
Nodo 9: x=[3. 3. 0. 0. 0. 0.], theta=0.0000, Q(x)=454770.9264, z_real=934770.9264
   ->

### 5.3 Verificación de factibilidad

In [13]:
cap_ok = True
dem_ok = True
for s in range(demanda.shape[0]):
    atendido_regular = demanda.to_numpy()[s] - u_opt[s]
    # cobertura total = atendido regularmente + emergencia debe igualar la demanda
    if not np.allclose(atendido_regular + u_opt[s], demanda.to_numpy()[s], atol=1e-4):
        dem_ok = False
print("¿Toda la demanda se cubre (regular + emergencia)?", dem_ok)
print("¿u* (emergencias) son todas >= 0?", bool(np.all(u_opt >= -1e-6)))
print("Capacidad instalada por zona (600*x_j):", (CAPACIDAD * x_opt).astype(int))

¿Toda la demanda se cubre (regular + emergencia)? True
¿u* (emergencias) son todas >= 0? True
Capacidad instalada por zona (600*x_j): [ 600  600 1200  600  600  600]


## 6. Conclusión

Se extendió el método L-Shaped de la Clase 3 (primera etapa continua) a **Integer L-Shaped**
(primera etapa entera), integrando Branch-and-Bound con la descomposición L-shaped y con
cortes globales (Benders y/o integralidad, según corresponda). La implementación se validó
en tres niveles de generalidad creciente:

| Validación | Dominio de $x$ | Recurso | Corte usado | Resultado |
|---|---|---|---|---|
| Ejemplo 1 (notas) | binario | entero | integralidad | coincide con el corte $\theta\ge0.75(x_2-x_1)-5.75$ del profesor y con la enumeración exhaustiva |
| Actividad 2 (notas) | binario | continuo | Benders | coincide con la iteración manual en $x^\nu=(1,0)$ y con la enumeración exhaustiva |
| Actividad 3 (Chicago) | entero general $\{0,\dots,3\}$ | continuo | Benders + ramificación | solución factible (demanda siempre cubierta, capacidad respetada) |

En los tres casos, cuando el recurso es continuo se generan cortes de Benders (iguales a los de
Clase 3) y, cuando $x$ es binaria, se generan además cortes de integralidad; para dominios enteros
generales (Actividad 3) el algoritmo se apoya únicamente en Benders + ramificación piso/techo, que
sigue siendo válido y exacto.